# ISOM 835 · Session 7 — Decision Trees & Random Forests
**Suffolk University · Sawyer Business School · Fall 2026 · Mon Nov 2 · Prof. Hasan Arslan**

**Tonight in one sentence:** our model learns to **ask questions like an experienced hotel manager**, and then we hire **300 of those managers** and let them vote.

**How to use this notebook:** run the cells top to bottom (Shift + Enter). Read the short text above each cell first: it tells you **what the cell does** and **how to read the output**. You do not need to understand every line of code tonight. You do need to understand every *result*. (The first code cell only checks your Python setup: run it and move on.)

🎮 **Play along:** [MLU-Explain: Decision Trees](https://mlu-explain.github.io/decision-tree/) grows a tree one question at a time, and [MLU-Explain: Random Forest](https://mlu-explain.github.io/random-forest/) shows the vote. Both work on your phone.

### Tonight's map

| Step | The question it answers |
|---|---|
| **0** | What are we predicting, and which columns would be cheating? |
| **1** | How does a tree ask questions? (a model you can print on one page) |
| **2** | How does a tree choose the *best* question? (Gini, in plain English) |
| **3** | What happens when a tree asks too many questions? (overfitting, again) |
| **4** | Why is one tree fragile? |
| **5** | How do 300 trees fix that? (random forests: the wisdom of the crowd) |
| **6** | Which clues really matter? |
| **7** | Which way does a clue push? (partial dependence) |
| **8** | How does all this connect to **neural networks**? |
| **9** | The leaderboard, and your turn |

Look for three kinds of boxes: **How to read it** (what an output means), **Stop and notice** (the lesson), and **🧠 Quick check** (a 20-second question to test yourself; click to see the answer).

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

## The business situation

You are the **revenue manager** for two hotels in Portugal: a **city hotel** in Lisbon and a **resort hotel** in the Algarve. **About 37% of bookings get canceled.** Every cancellation that comes too late to resell is an **empty room**: money lost.

Hotels fight back by **overbooking**: selling a few more rooms than they have, betting that some guests won't show up. Bet too little and rooms sit empty. Bet too much and a guest arrives to a full hotel; you pay for their room somewhere else and you lose a customer.

So you want one thing: **for each booking, how likely is it to be canceled?** That is tonight's prediction.

**The data:** real bookings from those two hotels (Antonio, de Almeida & Nunes, 2019). This is **the same data as our Kaggle competition**, so everything you learn tonight goes straight into your submission.

| Column | What it means |
|---|---|
| `is_canceled` | 1 = the booking was canceled (**our target**) |
| `lead_time` | days between making the booking and the arrival date |
| `deposit_type` | No Deposit, Non Refund, or Refundable |
| `previous_cancellations` | how many earlier bookings this guest canceled |
| `total_of_special_requests` | e.g. a high floor, a crib, a late check-in |
| `required_car_parking_spaces` | parking spaces the guest asked for |
| `market_segment` | how the booking came in (Online TA = online travel agent, Groups, Direct…) |
| `country` | the guest's country |
| `adr` | average daily rate: the price per night, in euros |

The next cell loads all the bookings and shows how many there are.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import roc_auc_score

# Load the bookings (the competition's training data, hosted in our course repo)
URL = 'https://raw.githubusercontent.com/harslan/isom-835/master/public/data/hotel_bookings.csv'
raw = pd.read_csv(URL)

print(f'{len(raw):,} bookings · {raw.shape[1]} columns · {raw["is_canceled"].mean():.0%} canceled')

## 0. Before any model: which columns are cheating?

A **leak** is a column that secretly contains the answer. A model that uses it gets a great score on paper and is useless in real life: like predicting a football game with **tomorrow's newspaper**.

Our data has one obvious leak: `reservation_status` (Canceled, Check-Out, No-Show). Look at it next to the target:

In [ ]:
# Rows: what reservation_status says. Columns: the target (0 = stayed, 1 = canceled).
pd.crosstab(raw['reservation_status'], raw['is_canceled'])

**How to read it:** every "Check-Out" booking has target 0 and every "Canceled" or "No-Show" booking has target 1. Not one exception. This column **is** the answer, written after the fact. We drop it, and two more:

| Column | Why it goes |
|---|---|
| `reservation_status` | the outcome in disguise (the table above) |
| `reservation_status_date` | the date of that outcome, also known only afterwards |
| `assigned_room_type` | the room the guest actually got **at check-in**. On the day you take the booking, you don't know it yet. |

The rule for every leak: **would I know this value at the moment I make the prediction?** If not, it goes. (The Kaggle competition files already have these three removed.)

### Getting the data ready for trees: much easier than before

In Sessions 3–6 the prep station had to **scale** numbers and **one-hot encode** text. **Trees need neither.** A tree only asks questions like *"is lead time more than 30 days?"*, and the answer is the same whether lead time is in days or in weeks. So the prep is just two things:

1. **Fill blanks** (with −1 for numbers, the word "missing" for text).
2. **Turn each text value into a code number**, with a small dictionary: City Hotel → 0, Resort Hotel → 1. That is `OrdinalEncoder`.

As always, the prep station learns its dictionary from the **training** bookings only. To keep class fast, we work with a random **30,000** bookings tonight (Your Turn Q3 shows how to use all of them for the competition).

In [ ]:
LEAKS = ['reservation_status', 'reservation_status_date', 'assigned_room_type']

bookings = raw.sample(30_000, random_state=835)            # a random 30,000 bookings, for speed in class
y = bookings['is_canceled']                                 # target: 1 = canceled
X_raw = bookings.drop(columns=['is_canceled'] + LEAKS)      # inputs: everything else, minus the leaks

# 80% to train, 20% held back as the exam (stratify keeps 37% canceled in both)
X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(X_raw, y, test_size=0.2, stratify=y, random_state=835)

def tree_prep(frame):
    # The prep station for trees: fill blanks, turn text into code numbers. No scaling, no one-hot.
    numbers = frame.select_dtypes(include='number').columns.tolist()
    text = [c for c in frame.columns if c not in numbers]
    return ColumnTransformer([
        ('numbers', SimpleImputer(strategy='constant', fill_value=-1), numbers),
        ('text', Pipeline([('fill', SimpleImputer(strategy='constant', fill_value='missing')),
                           ('codes', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), text),
    ], verbose_feature_names_out=False).set_output(transform='pandas')   # keep column names, so trees can show them

prep = tree_prep(X_raw)
X_tr = prep.fit_transform(X_tr_raw)     # learn the dictionary on the training bookings only
X_te = prep.transform(X_te_raw)         # apply the same dictionary to the test bookings

print(f'train: {len(X_tr):,} bookings · test: {len(X_te):,} bookings · {X_tr.shape[1]} columns, all numbers now\n')
codes = prep.named_transformers_['text'].named_steps['codes']
for col in ['hotel', 'deposit_type']:
    print(f'{col:13s} dictionary:', dict(enumerate(codes.categories_[list(codes.feature_names_in_).index(col)])))

### How we grade tonight: AUC (from Session 4)

Our model gives every booking a **probability of cancellation**. **AUC** asks: *if I pick one booking that canceled and one that didn't, how often does the model give the canceler the higher score?*

- **0.5** = a coin flip. **1.0** = it ranks every canceler above every stayer.
- AUC is also **the metric of our Kaggle competition**.

The next cell sets up a **leaderboard**. Every model tonight goes on it.

In [ ]:
leaderboard = []   # every model's test AUC goes here

def score(name, model, X_test=None):
    # Grade a fitted model on the test bookings and add it to the leaderboard
    X_test = X_te if X_test is None else X_test
    auc = roc_auc_score(y_te, model.predict_proba(X_test)[:, 1])
    leaderboard.append({'model': name, 'test AUC': auc})
    print(f'{name:40s} test AUC {auc:.3f}')

# The baseline: give every booking the same score. It cannot rank anyone above anyone.
print('Baseline: the same score for every booking  test AUC 0.500 (a coin flip)')
leaderboard.append({'model': 'Baseline: same score for everyone', 'test AUC': 0.5})

## 1. How a tree asks questions

Remember the game **20 Questions**? *"Is it an animal? Does it fly?"* Each answer cuts the possibilities down. A **decision tree** plays that game with your data: it learns a **flowchart of yes/no questions** that sorts bookings into groups that mostly cancel and groups that mostly stay.

To see the idea clearly, we start with a **manager's tree**: only **six clues** a hotel manager would recognize, and only **three questions deep**.

In [ ]:
# Six clues a manager would recognize, written in plain words
def manager_clues(X):
    return pd.DataFrame({
        'lead_time': X['lead_time'],                                          # days booked ahead
        'non_refund_deposit': (X['deposit_type'] == 'Non Refund').astype(int),  # 1 = paid a non-refundable deposit
        'special_requests': X['total_of_special_requests'],
        'previous_cancellations': X['previous_cancellations'],
        'parking_spaces': X['required_car_parking_spaces'],
        'online_travel_agent': (X['market_segment'] == 'Online TA').astype(int),
    })

M_tr, M_te = manager_clues(X_tr_raw), manager_clues(X_te_raw)

manager_tree = DecisionTreeClassifier(max_depth=3, random_state=835)   # at most 3 questions per booking
manager_tree.fit(M_tr, y_tr)

plt.figure(figsize=(20, 8))
plot_tree(manager_tree, feature_names=list(M_tr.columns), class_names=['stays', 'cancels'],
          filled=True, proportion=True, rounded=True, fontsize=11)
plt.title("The manager's tree: three questions, six clues", fontsize=15); plt.show()

score("Manager's tree (6 clues, 3 questions)", manager_tree, M_te)

**How to read one box:**
- **Line 1, the question.** `lead_time <= 14.5` means *"booked 14 days ahead or less?"* **Yes → go left. No → go right.** (For a 0/1 clue, `non_refund_deposit <= 0.5` means *"no non-refundable deposit?"*)
- **gini**: how mixed the box is (Step 2 explains it). 0 = everyone in the box did the same thing.
- **samples**: the share of all training bookings that land in this box.
- **value = [stays, cancels]**: the share of those bookings that stayed and that canceled.
- **color**: orange leans *stays*, blue leans *cancels*. Darker = more certain.

**Stop and notice:** read the tree out loud, like a manager would.
- **Non-refundable deposit?** About **99%** of those bookings cancel. That's backwards: why cancel after paying money you can't get back? Don't just use it, **ask why** (a great puzzle for your competition memo).
- **No such deposit, booked a week or less ahead?** Only about **9%** cancel. Last-minute guests show up.
- **Booked more than two weeks ahead and canceled before?** About **96%** cancel again. Past behavior predicts future behavior.

That is the whole cancellation story on one screen. **A model you can print and hand to a manager**: that's the superpower of a single tree.

## 2. How does the tree choose the *best* question?

At every box, the tree holds an **audition**: it tries every clue and every cut point (*lead time ≤ 1? ≤ 2? … ≤ 600?*) and keeps the question that makes the two new boxes **purest**.

**Gini impurity** measures how mixed a box is, in plain English:

> *Pick two guests at random from the box. What is the chance that **one cancels and the other doesn't**?*

- A box where everyone cancels (or everyone stays): Gini = **0**. Perfectly pure.
- A 50/50 box: Gini = **0.5**. As mixed as it gets.
- The formula is just that sentence: **Gini = 2 × p × (1 − p)**, where p is the share that cancels.

The next cell runs the audition for three candidate first questions.

In [ ]:
def gini(share_canceled):
    # chance that two random guests disagree: one cancels, the other doesn't
    p = share_canceled
    return 2 * p * (1 - p)

print(f'Before any question: {y_tr.mean():.0%} cancel → Gini = {gini(y_tr.mean()):.3f}\n')

candidates = {'non-refundable deposit?': M_tr['non_refund_deposit'] == 1,
              'canceled before?': M_tr['previous_cancellations'] > 0,
              'booked 7 days ahead or less?': M_tr['lead_time'] <= 7}

for question, yes in candidates.items():
    p_yes, p_no = y_tr[yes].mean(), y_tr[~yes].mean()
    share_yes = yes.mean()
    after = share_yes * gini(p_yes) + (1 - share_yes) * gini(p_no)     # weighted by box size
    print(f'{question:30s} YES box: {p_yes:4.0%} cancel   NO box: {p_no:4.0%} cancel   '
          f'Gini after = {after:.3f}  (drop {gini(y_tr.mean()) - after:.3f})')

**How to read it:** each question splits the bookings into a YES box and a NO box. "Gini after" is the average mixedness of the two boxes (bigger boxes count more). The **biggest drop wins**, and that question goes at the top of the tree. Then the tree repeats the audition inside each new box.

**The tree is greedy:** it picks the best question **right now**, without planning ahead. Fast and usually good, but not guaranteed to find the best possible tree.

### 🧠 Quick check

Box A: 90% of bookings cancel. Box B: 50% cancel. Which box has the higher Gini, and what does it mean?

<details><summary>Click for the answer</summary>

**Box B.** Gini A = 2 × 0.9 × 0.1 = **0.18**; Gini B = 2 × 0.5 × 0.5 = **0.50**. In Box B two random guests disagree half the time: the box tells you almost nothing. Box A is close to pure: "these bookings cancel." The tree is hunting for boxes like A.
</details>

## 3. All the clues, and a tree that asks too many questions

Now give the tree **all 28 columns** and let it ask more questions. How deep should it go? **Depth** = the number of questions from the top to a final box (a **leaf**).

Remember the **memorizer** from Session 6, the student with the answer key? A tree can do the same thing: keep asking questions until **every leaf holds just one or two training bookings**. Perfect on practice, lost on the exam. The next cell grows trees from 2 to unlimited depth and grades each on **both** the training bookings and the test bookings.

In [ ]:
depths = [2, 4, 6, 8, 10, 14, 20, None]     # None = no limit: keep asking until every leaf is pure
rows = []
for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=835).fit(X_tr, y_tr)
    rows.append({'max_depth': 'no limit' if depth is None else depth,
                 'leaves': tree.get_n_leaves(),
                 'train AUC': roc_auc_score(y_tr, tree.predict_proba(X_tr)[:, 1]),
                 'test AUC': roc_auc_score(y_te, tree.predict_proba(X_te)[:, 1])})
curve = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8, 4))
labels = curve['max_depth'].astype(str)
ax.plot(labels, curve['train AUC'], 'o-', color='#5b6bd8', lw=2, label='practice test (training bookings)')
ax.plot(labels, curve['test AUC'], 'o-', color='#d64545', lw=2, label='real exam (test bookings)')
best = curve['test AUC'].idxmax()
ax.annotate('sweet spot', (best, curve['test AUC'][best]), xytext=(0, -38), textcoords='offset points',
            ha='center', color='#2e8b57', fontweight='bold', arrowprops=dict(arrowstyle='->', color='#2e8b57'))
ax.set_xlabel('max_depth (how many questions the tree may ask)'); ax.set_ylabel('AUC')
ax.set_title('Deeper trees: better on practice, worse on the exam'); ax.legend(loc='lower right'); plt.show()

curve.round(3)

**Stop and notice:**
- **Practice score (blue) keeps climbing.** With no limit, the tree grows about **3,000 leaves** for 24,000 bookings and gets a **perfect 1.00** on the bookings it trained on.
- **Exam score (red) rises, peaks around depth 10 (≈ 0.91), then falls** to about 0.81. The extra questions learned the quirks of individual bookings, not patterns of cancellation.
- The **gap** between the two lines is overfitting you can see.

So `max_depth` is a **brake**. Here are the two trees worth remembering, on the leaderboard:

In [ ]:
best_depth_tree = DecisionTreeClassifier(max_depth=10, random_state=835).fit(X_tr, y_tr)
unlimited_tree = DecisionTreeClassifier(random_state=835).fit(X_tr, y_tr)

score('One tree, no limit (the memorizer)', unlimited_tree)
score('One tree, max_depth = 10', best_depth_tree)

### A second brake: `min_samples_leaf`

`min_samples_leaf=50` means: **every final box must hold at least 50 training bookings.** A manager would never write a policy based on one guest; this brake says the same thing to the tree. It is often a better brake than depth for business data, because the tree can still go deep where there is lots of data, and stays cautious where there isn't.

In [ ]:
for leaf in [1, 5, 20, 50, 200]:
    tree = DecisionTreeClassifier(min_samples_leaf=leaf, random_state=835).fit(X_tr, y_tr)
    test_auc = roc_auc_score(y_te, tree.predict_proba(X_te)[:, 1])
    print(f'every leaf holds at least {leaf:3d} bookings → {tree.get_n_leaves():5,d} leaves   test AUC {test_auc:.3f}')

leaf50_tree = DecisionTreeClassifier(min_samples_leaf=50, random_state=835).fit(X_tr, y_tr)
print()
score('One tree, min_samples_leaf = 50', leaf50_tree)
best_tree_auc = leaderboard[-1]['test AUC']   # we draw this line again in Step 5

**Stop and notice:** with "at least 50 bookings per leaf" the tree keeps about **270 leaves** instead of about 3,000, and scores **≈ 0.92**: better than the best depth. Too strict (200) and it starts to miss real patterns: the brake works like the ridge tax in Session 6, too little and it memorizes, too much and it underfits.

### 🧠 Quick check

A tree scores **0.99** AUC on its training bookings and **0.80** on the test bookings. What happened, and name one setting you would change.

<details><summary>Click for the answer</summary>

It **memorized** the training bookings (overfitting): the big gap between practice and exam gives it away. Put on a brake: a smaller `max_depth` (say 8–10) or a bigger `min_samples_leaf` (say 20–50).
</details>

## 4. Why one tree is fragile

Here is the hidden weakness of a single tree. We split our training bookings into **two random halves** and grow one tree on each. Same hotel, same kind of guests, just a different random half. Do the two trees agree about the test bookings?

In [ ]:
shuffled = np.random.default_rng(835).permutation(len(X_tr))
half_A, half_B = shuffled[: len(X_tr) // 2], shuffled[len(X_tr) // 2 :]

tree_A = DecisionTreeClassifier(random_state=835).fit(X_tr.iloc[half_A], y_tr.iloc[half_A])
tree_B = DecisionTreeClassifier(random_state=835).fit(X_tr.iloc[half_B], y_tr.iloc[half_B])

trees_disagree = (tree_A.predict(X_te) != tree_B.predict(X_te)).mean()
print(f'Two trees, two random halves of the same data: they disagree on {trees_disagree:.0%} of the test bookings')

**Stop and notice:** about **1 booking in 5** gets *"will cancel"* from one tree and *"will stay"* from the other. Nothing changed except **which random guests each tree happened to see**.

That's what **high variance** means: the model's answers swing with the luck of the data. It's like asking **one** colleague for advice: their answer depends a lot on the few cases they happen to remember.

**The fix: don't ask one colleague. Ask a crowd.**

## 5. From one tree to a forest: the wisdom of the crowd

In 1906 the scientist **Francis Galton** watched about **800 people** at a country fair guess the weight of an ox. Most guesses were far off, but the **middle guess was within 1% of the true weight**: better than the cattle experts. Individual errors pointed in different directions and **cancelled out**.

A **random forest** builds that crowd out of trees, with two tricks so the trees make *different* mistakes:

1. **Each tree studies a different sample of bookings.** Draw bookings at random **with replacement**, like pulling names from a hat and putting each name back: some bookings appear twice, about a third not at all. (This is called a **bootstrap sample**; averaging trees built this way is called **bagging**.)
2. **At every question, each tree may only look at a random handful of clues.** One tree can't lean on `deposit_type` every time; it is forced to find other patterns.

Then every tree votes, and the forest **averages the votes** into a probability.

> **Why trick 2?** If 300 advisors all read the same newspaper, the crowd is no wiser than one of them. Forcing the trees to look at different clues makes their mistakes **different**, and different mistakes cancel.

The next cell builds both: **bagging** (trick 1 only) and a **random forest** (tricks 1 and 2). It takes about 20–30 seconds.

In [ ]:
# Trick 1 only: 100 unlimited trees, each on its own bootstrap sample
bagging = BaggingClassifier(DecisionTreeClassifier(random_state=835), n_estimators=100, n_jobs=-1, random_state=835)
bagging.fit(X_tr, y_tr)

# Tricks 1 + 2: a random forest of 300 trees; each leaf holds at least 5 bookings; oob_score explained below
forest = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, oob_score=True, n_jobs=-1, random_state=835)
forest.fit(X_tr, y_tr)

score('Bagging: 100 trees, bootstrap samples', bagging)
score('Random forest: 300 trees', forest)

**Stop and notice:** the best single tree scored about **0.92**. The crowd scores about **0.94**, with no tuning, no scaling, and the **memorizer trees** inside the bag (each one unlimited!). Averaging turned fragile trees into a strong model.

On this data bagging and the forest are almost tied: trick 1 does most of the work here. Trick 2 matters more when a few very strong clues would otherwise sit at the top of every tree, and it also makes each tree faster to grow. Now the fragility test from Step 4, again, with forests instead of trees:

In [ ]:
forest_A = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=835).fit(X_tr.iloc[half_A], y_tr.iloc[half_A])
forest_B = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=835).fit(X_tr.iloc[half_B], y_tr.iloc[half_B])

forests_disagree = (forest_A.predict(X_te) != forest_B.predict(X_te)).mean()
print(f'Two single trees, two random halves:  disagree on {trees_disagree:.0%} of test bookings')
print(f'Two forests,      two random halves:  disagree on {forests_disagree:.0%} of test bookings')

**Stop and notice:** from about **1 in 5** down to about **1 in 25**. The forest's answers no longer depend much on which guests it happened to see. **That is what "reducing variance" means**, and you just watched it happen.

### How many trees?

More trees never make a forest memorize more; they just make the vote **steadier**, and the forest slower. The next cell grows one forest tree by tree and checks the exam score along the way.

In [ ]:
growing = RandomForestClassifier(min_samples_leaf=5, n_jobs=-1, random_state=835, warm_start=True)   # warm_start: keep the trees, add more
sizes, aucs = [1, 2, 5, 10, 25, 50, 100, 200, 300], []
for n in sizes:
    growing.set_params(n_estimators=n).fit(X_tr, y_tr)
    aucs.append(roc_auc_score(y_te, growing.predict_proba(X_te)[:, 1]))

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot([str(n) for n in sizes], aucs, 'o-', color='#2e8b57', lw=2)
ax.axhline(best_tree_auc, color='#999999', ls='--', label='best single tree (min_samples_leaf = 50)')
ax.set_xlabel('number of trees in the forest'); ax.set_ylabel('test AUC')
ax.set_title('Adding trees helps fast, then flattens: it never hurts'); ax.legend(loc='lower right'); plt.show()
print('  '.join(f'{n} trees: {a:.3f}' for n, a in zip(sizes, aucs)))

**How to read it:** one tree alone is weak (≈ 0.86). **Five** trees already beat the best single tree (the dashed line). After about **100 trees** the line is flat. So `n_estimators=300` is a safe default: it rarely helps much past that, and it **never hurts** except in waiting time.

### A free exam: the out-of-bag score

Remember: each tree's bootstrap sample **left out about a third of the bookings** (the names that never came out of the hat). So for every training booking, about 100 of our 300 trees **never saw it**. Ask only those trees about that booking, and you get an honest grade **without touching the test set**. That's the **out-of-bag (OOB)** score, and we asked for it with `oob_score=True`.

In [ ]:
oob_auc = roc_auc_score(y_tr, forest.oob_decision_function_[:, 1])   # each training booking, graded only by trees that never saw it
test_auc = roc_auc_score(y_te, forest.predict_proba(X_te)[:, 1])
print(f'Out-of-bag AUC (no test set needed): {oob_auc:.3f}')
print(f'Test AUC (the real exam):            {test_auc:.3f}')

**Stop and notice:** the free exam and the real exam agree closely. When data is scarce, the OOB score lets you **keep every row for training** and still get an honest grade.

### 🧠 Quick check

Your colleague says: *"My forest has 1,000 trees, so it must be overfitting. Cut it to 50."* Is that right?

<details><summary>Click for the answer</summary>

**No.** Adding trees makes the vote steadier; it does not make the forest memorize more (look at the flat line above). 1,000 trees is just slower. If the forest overfits, the brakes are on each tree: `min_samples_leaf`, `max_depth`, or how many clues each question may look at (`max_features`).
</details>

## 6. Which clues matter? Two answers that disagree

Your manager will ask: **"What drives cancellations?"** Forests give two different answers, and one of them can fool you.

- **Impurity importance** (`feature_importances_`): how much each clue **reduced Gini** inside the trees, on the **training** bookings.
- **Permutation importance**: take the **test** bookings and **shuffle one column** (like scrambling one line on every guest's file). If the forest's AUC drops a lot, that clue mattered. If nothing changes, the forest never really needed it.

**The trap:** a column with thousands of different values (like an ID number) offers **thousands of places to cut**. Try enough cuts and some will look useful **by pure luck** on the training data. So we plant two **fake clues** that cannot possibly predict cancellation:
- `noise`: random numbers;
- `booking_id`: a made-up ID number, different for every booking.

Then we ask both methods where they rank. (About 30–60 seconds: permutation importance re-scores the forest many times.)

In [ ]:
rng = np.random.default_rng(835)
X_tr_fake = X_tr.assign(noise=rng.normal(size=len(X_tr)), booking_id=rng.permutation(len(X_tr)))
X_te_fake = X_te.assign(noise=rng.normal(size=len(X_te)), booking_id=rng.permutation(len(X_te)) + len(X_tr))

forest_fake = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=835).fit(X_tr_fake, y_tr)

importance = pd.DataFrame({'impurity (training)': forest_fake.feature_importances_}, index=X_tr_fake.columns)
shuffled = permutation_importance(forest_fake, X_te_fake, y_te, scoring='roc_auc', n_repeats=5, random_state=835, n_jobs=-1)
importance['permutation (test)'] = shuffled.importances_mean

ranks = importance.rank(ascending=False).astype(int)
for fake in ['noise', 'booking_id']:
    print(f"{fake:11s} rank by impurity: #{ranks.loc[fake, 'impurity (training)']:2d}    "
          f"rank by permutation: #{ranks.loc[fake, 'permutation (test)']:2d}    (of {len(importance)} columns)")

# Side by side: the top 12 by impurity, with the two fakes in red
top = importance.sort_values('impurity (training)', ascending=False).head(12)
colors = ['#d64545' if c in ('noise', 'booking_id') else '#5b6bd8' for c in top.index]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
ax[0].barh(top.index, top['impurity (training)'], color=colors); ax[0].set_title('Impurity importance (training)')
ax[1].barh(top.index, top['permutation (test)'], color=colors); ax[1].set_title('Permutation importance (test)\n= drop in AUC when the column is shuffled')
ax[0].invert_yaxis(); plt.tight_layout(); plt.show()

**Stop and notice:**
- **By impurity**, the two fake clues land around **#10 and #11 of 30**, in the top third, **ahead of real signals** like parking spaces and customer type.
- **By permutation**, they sink to the **bottom three** (#28 and #29 of 30): shuffling them changes nothing, because they never predicted anything.
- The real story, by permutation: **country, deposit type, special requests, market segment, lead time**.

**The rule:** impurity importance rewards columns with **many split points**, not columns that **predict**. If you report "what drives cancellations" to a manager, **report permutation importance on test data.**

## 7. Partial dependence: *which way* does a clue push?

Importance says **how much** a clue matters. It doesn't say **which way**. Do long lead times mean *more* cancellations or fewer? Is the effect a straight line or a curve?

A **partial dependence plot** answers with a simple experiment: take 3,000 test bookings, **pretend every one of them was booked 0 days ahead**, and average the forest's predicted cancellation chance. Then pretend 10 days, 20 days, and so on. Everything else about each booking stays as it really was.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.8))
PartialDependenceDisplay.from_estimator(forest, X_te.sample(3000, random_state=835),
                                        ['lead_time', 'total_of_special_requests', 'previous_cancellations'],
                                        kind='average', ax=ax)
fig.suptitle('Predicted chance of cancellation as one clue changes, everything else held as it was', y=1.02)
plt.tight_layout(); plt.show()

**How to read it:** the line is the forest's **average predicted chance of cancellation** (up the side) as one clue changes (along the bottom). The tick marks at the bottom show where real bookings are.

**Stop and notice:**
- **Lead time:** booked today, about **22%** cancel. By about **one month ahead** it is already about **39%**, then it creeps up slowly to about **48%** for bookings made 10 months ahead. The danger zone starts early. **A curve a revenue manager can price**, e.g. deposit rules that start at 30 days.
- **Special requests:** zero requests, about **45%**; one or more, about **27%**. Guests who ask for a crib or a quiet room are planning to come.
- **Previous cancellations:** none, about **36%**; one or more, about **66%**.

> ⚠️ **Same warning as Session 6: describe, don't claim causes.** "Guests who make special requests cancel less" is a pattern. It does **not** mean that asking every guest to make a request would cut cancellations.

## 8. The bridge to neural networks

You will hear about **neural networks** everywhere (ChatGPT is one). Trees and neural networks are different families, but tonight's ideas are the **best doorway** into them. Four connections.

### Connection 1: a tree asks about one clue; a neuron asks about a *mix* of clues

| | A tree's question | A neuron's question |
|---|---|---|
| Example | *"Is lead time more than 14 days?"* | *"Is 0.8 × lead time − 0.3 × price + 2 × special requests more than 5?"* |
| Looks at | **one** column | a **weighted mix** of all columns |
| Answer | a hard **yes / no** | a **soft** "how much yes", between 0 and 1 |
| How it's learned | greedy: the best question, one box at a time | all the weights nudged together, little by little (**gradient descent**) |

A **neural network** is layers of these soft questions: the answers of the first layer become the clues for the next layer. That is how it can invent its *own* clues, such as *"a long-lead, cheap, no-request booking"*, instead of using our columns as they are.

### Connection 2: how each model carves up the map

The best way to *see* the difference: give each model just **two clues**, lead time and price per night, and color every spot on the map by the predicted chance of cancellation (**blue = stays, red = cancels**).

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder

two = ['lead_time', 'adr']                      # just two clues, so we can draw the map
A_tr, A_te = X_tr[two], X_te[two]

map_models = {
    'One tree (depth 6)': DecisionTreeClassifier(max_depth=6, random_state=835),
    'Random forest (300 trees)': RandomForestClassifier(n_estimators=300, min_samples_leaf=20, n_jobs=-1, random_state=835),
    'Neural network (2 layers of 32 neurons)': Pipeline([('scale', StandardScaler()),
        ('net', MLPClassifier(hidden_layer_sizes=(32, 32), early_stopping=True, max_iter=500, random_state=835))]),
}

lead, price = np.meshgrid(np.linspace(0, 400, 200), np.linspace(0, 250, 200))
grid = pd.DataFrame({'lead_time': lead.ravel(), 'adr': price.ravel()})

fig, ax = plt.subplots(1, 3, figsize=(16, 4.6), sharey=True)
for a, (name, model) in zip(ax, map_models.items()):
    model.fit(A_tr, y_tr)
    chance = model.predict_proba(grid)[:, 1].reshape(lead.shape)
    shade = a.contourf(lead, price, chance, levels=np.linspace(0, 1, 21), cmap='coolwarm')
    a.set_title(f'{name}\ntest AUC {roc_auc_score(y_te, model.predict_proba(A_te)[:, 1]):.3f}')
    a.set_xlabel('lead time (days booked ahead)')
ax[0].set_ylabel('price per night (€)')
fig.colorbar(shade, ax=ax, label='predicted chance of cancellation'); plt.show()

**How to read it:** each panel is a map of every possible booking with these two clues. Where the color is red, the model expects a cancellation.

**Stop and notice:**
- **The tree** carves the map into **rectangles**: every question is a straight cut along one axis (*"lead time > 14?"*, *"price > 90?"*).
- **The forest** averages 300 different sets of rectangles: a **finer patchwork** with softer edges. That averaging is why it scores higher.
- **The neural network** draws **smooth curves**, because each neuron cuts along a *mix* of the two clues. Different shapes, same goal: put the red where the cancellations are.

### Connection 3: the same lessons carry over

| Tonight, with trees | In a neural network |
|---|---|
| A deep tree **memorizes** (Step 3) | A big network memorizes too: millions of knobs |
| Brakes: `max_depth`, `min_samples_leaf` | Brakes: **early stopping**, weight decay (a complexity tax, like ridge in Session 6) |
| A forest's random handful of clues makes trees **different**, and averaging cancels mistakes | **Dropout**: during training, randomly switch off neurons. Its inventors describe it as averaging many "thinned" networks: a crowd hiding inside one network |
| Permutation importance and partial dependence explain the forest | They work on **any** model, neural networks included: just shuffle a column and watch |

### Connection 4: head to head on our hotel table

A neural network needs the **full prep station** from Sessions 3–6: numbers on the same scale, text one-hot encoded. Watch what happens without it. (About 20 seconds.)

In [ ]:
num_cols = X_tr_raw.select_dtypes(include='number').columns.tolist()
text_cols = [c for c in X_tr_raw.columns if c not in num_cols]

# The Session 3 prep station: fill, scale, one-hot. Neural networks need all of it.
net_prep = ColumnTransformer([
    ('numbers', Pipeline([('fill', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num_cols),
    ('text', Pipeline([('fill', SimpleImputer(strategy='constant', fill_value='missing')),
                       ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=20))]), text_cols)])
net = Pipeline([('prep', net_prep),
                ('net', MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True, max_iter=300, random_state=835))])
net.fit(X_tr_raw, y_tr)

# The same network, fed the tree-style data: raw numbers and code numbers, no scaling
lazy_net = MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True, max_iter=300, random_state=835).fit(X_tr, y_tr)

print(f"Neural network, tree-style data (no scaling):  test AUC {roc_auc_score(y_te, lazy_net.predict_proba(X_te)[:, 1]):.3f}")
score('Neural network (scaled + one-hot)', net, X_te_raw)
print(f"Random forest, no scaling needed:              test AUC {roc_auc_score(y_te, forest.predict_proba(X_te)[:, 1]):.3f}")

**Stop and notice:**
- Without scaling, the network is badly hurt (≈ **0.86**): lead time runs up to 600 and `is_repeated_guest` only to 1, and the network's weighted mixes are thrown off by the big numbers. **The forest never cared** (Step 0).
- With the full prep, the network reaches about **0.93**: good, but **still behind the forest** (≈ 0.94), which needed no prep and no tuning.
- This is not a fluke. A 2022 study across 45 datasets, *"Why do tree-based models still outperform deep learning on typical tabular data?"* (Grinsztajn, Oyallon & Varoquaux), found the same: **on tables, tree ensembles still usually win.** On images, sound and text, neural networks win by a mile (Session 12).

> **The big picture:** a forest is the **wisdom of a crowd of simple experts**; a neural network is **one expert that invents its own clues**. Next week you'll meet the newest twist: neural networks trained on millions of *fake tables* (TabPFN) that are starting to beat trees on small tables.

## The leaderboard

Everything we built tonight, on the same 6,000 test bookings (the two-clue map models are not on it: they only saw two columns).

In [ ]:
board = pd.DataFrame(leaderboard).drop_duplicates('model', keep='last').set_index('model')

colors = ['#999999' if m.startswith('Baseline') else '#2e8b57' if m == board['test AUC'].idxmax() else '#5b6bd8' for m in board.index]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(board.index, board['test AUC'], color=colors)
for row, auc in enumerate(board['test AUC']):
    ax.text(auc + 0.005, row, f'{auc:.3f}', va='center')
ax.invert_yaxis(); ax.set_xlim(0.45, 1.0); ax.set_xlabel('test AUC (higher is better)')
ax.set_title('Tonight, from a coin flip to a forest'); plt.show()

### One tree or a forest? When to use which

| | **One decision tree** | **Random forest** |
|---|---|---|
| Accuracy | good with a brake (≈ 0.92 here) | **better** (≈ 0.94 here), with no tuning |
| Explain it | **print it on one page** | explain it with permutation importance + partial dependence |
| Stable? | no: a different sample, a different tree | **yes**: the vote is steady |
| Scaling or one-hot needed? | **no** | **no** |
| Use it when… | you need a rule a person can follow | you need the best predictions from table data, fast |

Next week, **gradient boosting**: trees that learn from each other's mistakes instead of voting. It is probably the model that wins our competition.

## 9. Your turn

Each question has a starter cell. Fill in the line marked `# ← your turn`, run it, then answer in words. Work with a neighbor if you like.

1. **Leak test.** Put `assigned_room_type` back in. How much does the forest's AUC change? Would you know this column when the booking is made?
2. **Two brakes at once.** Combine `max_depth` and `min_samples_leaf` in one tree. Can a single tree beat **0.916**?
3. **Competition.** Put a random forest into the Competition Starter notebook and submit tonight.
4. **Challenge (optional).** The decorrelation dial: `max_features`.

**Q1 · Leak test.** The starter rebuilds the prep station with `assigned_room_type` included. One line to finish: fit the forest.

In [ ]:
# Q1 · the forest with the check-in column put back
X_leak_raw = bookings.drop(columns=['is_canceled', 'reservation_status', 'reservation_status_date'])   # assigned_room_type stays in
Xl_tr_raw, Xl_te_raw = X_leak_raw.loc[X_tr_raw.index], X_leak_raw.loc[X_te_raw.index]                # same train/test bookings as before

prep_leak = tree_prep(X_leak_raw)
Xl_tr, Xl_te = prep_leak.fit_transform(Xl_tr_raw), prep_leak.transform(Xl_te_raw)

forest_leak = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=835)
...   # ← your turn: fit forest_leak on Xl_tr and y_tr

print(f'Without assigned_room_type: test AUC {roc_auc_score(y_te, forest.predict_proba(X_te)[:, 1]):.3f}')
print(f'With assigned_room_type:    test AUC {roc_auc_score(y_te, forest_leak.predict_proba(Xl_te)[:, 1]):.3f}')
print(f"\nCancel rate when the guest got a different room than booked: {bookings.loc[bookings['reserved_room_type'] != bookings['assigned_room_type'], 'is_canceled'].mean():.0%}")
print(f"Cancel rate when the guest got the room they booked:         {bookings.loc[bookings['reserved_room_type'] == bookings['assigned_room_type'], 'is_canceled'].mean():.0%}")

Your answer to Q1 (double-click to edit): *the AUC changed by about \_\_\_. I would / would not keep this column because…* (Hint: look at the two cancel rates. Why would a guest who got a *different* room almost never cancel? When is the room assigned? And if a leak barely moves the score, is it still a leak?)

**Q2 · Two brakes at once.** Try a few combinations. Which pair gives the best single tree?

In [ ]:
# Q2 · combine the two brakes
for depth in [8, 10, 14]:
    for leaf in [20, 50]:          # ← your turn: try other values too, e.g. 10 or 100
        tree = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf, random_state=835).fit(X_tr, y_tr)
        print(f'max_depth {depth:2d}, min_samples_leaf {leaf:3d}: {tree.get_n_leaves():4d} leaves   '
              f'test AUC {roc_auc_score(y_te, tree.predict_proba(X_te)[:, 1]):.3f}')

Your answer to Q2: *the best single tree uses max_depth = \_\_\_ and min_samples_leaf = \_\_\_, AUC \_\_\_. It still loses to the forest by about \_\_\_.*

**Q3 · Competition: get a forest on the leaderboard tonight.** Open the [Competition Starter](https://colab.research.google.com/github/harslan/isom-835/blob/master/public/session-05/ISOM835_Competition_Starter.ipynb) (or your own copy). In **Section 2**, replace the cell that builds `prep` and `model` with the cell below. Then run Section 3 as it is: the starter's `X`, `y` and `test` are already defined by then.

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder

num = X.select_dtypes(include='number').columns.tolist()
cat = [c for c in X.columns if c not in num]
prep = ColumnTransformer([
    ('num', SimpleImputer(strategy='constant', fill_value=-1), num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')),
                      ('codes', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), cat),
])
model = Pipeline([('prep', prep), ('clf', RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=835))])
cv = StratifiedKFold(5, shuffle=True, random_state=835)
scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
print(f'5-fold AUC: {scores.mean():.4f} ± {scores.std():.4f}')
```

On all **83,573** training bookings, our check gives a 5-fold AUC of about **0.95**, up from about **0.89** for the logistic baseline. Five-fold cross-validation trains five forests, so in Colab it takes a few minutes; start it and keep reading. Then run Section 3 to write `submission.csv`, submit it, and **upload the same file on Canvas**.

Your answer to Q3: *my 5-fold AUC was \_\_\_; the public leaderboard said \_\_\_. They differ because…*

**Q4 · Challenge (optional): the decorrelation dial.** `max_features` is trick 2: how many clues each question may look at. `'sqrt'` (the default) means about 5 of our 28; `None` means all 28, which turns the forest back into plain bagging.

In [ ]:
# Q4 · how much does forcing trees to look at different clues help here?
for clues in ['sqrt', 0.5, None]:      # ← your turn: try 2 or 3 as well (a whole number = that many clues)
    m = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, max_features=clues, n_jobs=-1, random_state=835).fit(X_tr, y_tr)
    print(f'max_features = {str(clues):5s}  test AUC {roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]):.3f}')
# Which setting won? Would you expect a bigger difference on data where one clue dominates? One sentence below.

## What we learned tonight

- A **decision tree** is a **flowchart of yes/no questions learned from data**. It picks each question by how much it **reduces Gini** (the chance two random guests disagree). It needs **no scaling and no one-hot**, and you can **print it for a manager**.
- A tree that asks too many questions **memorizes**. **`max_depth`** and **`min_samples_leaf`** are its brakes.
- One tree is **fragile**: a different sample, a different tree. A **random forest** averages hundreds of trees, each on a **bootstrap sample** and each question using a **random handful of clues**. Different mistakes cancel out: the **wisdom of the crowd**. More trees never hurt. The **out-of-bag score** is a free exam.
- **Impurity importance can be fooled** by ID-like columns. **Report permutation importance on test data**, and use **partial dependence** to show which way a clue pushes.
- **The bridge to neural networks:** a tree asks about **one clue** with a hard yes/no; a neuron asks about a **weighted mix** of clues with a soft answer. The same lessons carry over: big models memorize, brakes help, crowds (forests, dropout) cancel mistakes. On **tables**, forests still usually win, and they need no scaling; neural networks shine on images, sound and text.

**Tonight, Mon Nov 2, 11:59 PM:** Project Milestone 1 (your problem and dataset, one paragraph). **Competition:** at least one random-forest submission from your team this week; the leaderboard closes **Mon Nov 9, 4:30 PM** (upload the same predictions file on Canvas by then), and the one-page memo is due that night, **11:59 PM**. Keep a running list of every column you dropped as a leak, and why. **HW3** is due **Mon Nov 16**. **Reading:** ISLP 8.2.3 (Boosting) before next class.

**Next week:** gradient boosting, the model that will probably win, and the new tabular foundation models that might beat it.